# 1a · How many tests are needed to be confident in a displayed IQB score?

**Research question (RQ 1a)** *"Conduct bootstrap resampling to establish minimum measurement thresholds for score reliability."*

In order to display a country's IQB score as a single number (to one decimal), we posit that the score must be **±0.05** of the displayed score with at least **95%** probability. This follows the guidelines also set forth in Ookla Speedtest Global Index. 


In [ ]:
import numpy as np
import pandas as pd

import iqb_tools as T

T.set_style()

cfg = T.config_table()  # IQB library default thresholds
fw_min = T.with_thresholds(cfg, T.FRAMEWORK_MIN)  # IQB framework, minimum quality
fw_high = T.with_thresholds(cfg, T.FRAMEWORK_HIGH)  # IQB framework, high quality
df = T.load_level("country")
print(
    f"{len(df):,} country-months, {df.country_code.nunique()} countries, "
    f"{df.month.min():%b %Y} – {df.month.max():%b %Y}"
)

The printout describes the data loaded: one row per country and month (a *country-month*), 5,720 in all, for 245 countries over the 24 months from January 2024 to December 2025. Not every country has tests in every month, so there are fewer rows than 245 × 24 = 5,880.

## How many tests? 

IQB score at p50 describes the typical measured connection where half of the speed tests in the country met or exceeded threshold values. Whether that score can be trusted depends on how far the pass rate is from 50%: the closer it is, the more tests are needed to be sure which side of 50% it falls on (i.e. pass or fail). That number grows with the inverse square of the distance from 50%.

Let's take one row from the download data (`downloads_by_country`, November 2025):

|column|value|
|---|---|
| country_code | AD |
| sample_count | 2001 |
| download_p1 | 1.55 |
| download_p5 | 12.08 |
| download_p10 | 22.78 |
| download_p25 | 52.33 |
| download_p50 | 122.22 |
| download_p75 | 286.33 |
| download_p90 | 481.80 |
| download_p95 | 635.13 |
| download_p99 | 859.83 |

Given the threshold of 100 Mbps for download, 100 Mbps falls between `download_p25` (52.33 Mbps) and `download_p50` (122.22 Mbps).

As the data provides only nine percentile values (p1–p99) per country rather than the full cumulative distribution, we construct a continuous CDF by interpolating between the values, which assumes tests are spread evenly between adjacent percentiles.

### Which interpolation scale?

Between two published percentiles, the shape of the distribution is unknown. We compare linear and logarithmic interpolation with a held-out-percentile test, illustrated below for Andorra's download speeds in November 2025.

**Step 1: Hide one percentile.** Each interior percentile is removed in turn and treated as unknown. For Andorra, hiding $P_{25}$ leaves its neighbors $P_{10}$ and $P_{50}$:

| | $P_{10}$ | $P_{25}$ (hidden) | $P_{50}$ |
|---|---|---|---|
| Download (Mbps) | 22.78 | 52.33 | 122.22 |

**Step 2: Predict it from its neighbors.** The hidden value at level $p$ is predicted from the neighboring values $q_{\text{lo}}$ and $q_{\text{hi}}$ at levels $p_{\text{lo}}$ and $p_{\text{hi}}$. Writing $w = (p - p_{\text{lo}})/(p_{\text{hi}} - p_{\text{lo}})$:

$$
\hat{q}_{\text{linear}} = q_{\text{lo}} + w\,(q_{\text{hi}} - q_{\text{lo}})
$$

$$
\hat{q}_{\text{log}} = \exp\!\big(\ln q_{\text{lo}} + w\,(\ln q_{\text{hi}} - \ln q_{\text{lo}})\big)
$$

For $P_{25}$, $w = (0.25 - 0.10)/(0.50 - 0.10) = 0.375$, giving $\hat{q}_{\text{linear}} \approx 60.1$ Mbps and $\hat{q}_{\text{log}} \approx 42.8$ Mbps, against a true value of 52.33 Mbps.

**Step 3: Score the prediction against each threshold.** Because IQB uses only whether a value passes or fails a threshold, a prediction is scored as correct when it lies on the same side of a threshold as the true value. Every IQB threshold between the two neighbors is checked. For Andorra these are the download thresholds of 25, 50 and 100 Mbps:

| Threshold (Mbps) | True $P_{25}$ (52.3) | Linear (60.1) | Log (42.8) |
|---|---|---|---|
| 25 | above | above ✓ | above ✓ |
| 50 | above | above ✓ | below ✗ |
| 100 | below | below ✓ | below ✓ |


As can be seen in the table above, linear interpolation predicts the p25 as 60.1 Mbps whereas log interpolation predicts the p25 value as 42.8 Mbps. If the threshold to clear for download speed had been 25 Mbps, both linear and logarithmic interpolation values would correctly place the predicted p25 value as above 25 Mbps, and both interpolations therefore "pass" the download speed as meeting the required threshold. 

However, when taking 50 Mbps as the threshold, linear interpolation predicts 60.1 Mbps. This agrees with the true p25 value and "passes" the required threshold. Conversely, the log interpolation predicts a p25 value of 42.8 Mbps, thus not clearing the required 50 Mbps for the download threshold. This "fails" the download requirement at p25, though we can see that the true value satisfies the given threshold.  

In this single case linear interpolation is correct for all three thresholds and logarithmic interpolation is correct for two.

**Step 4: Repeat across all cases.** The procedure is repeated for every interior percentile ($P_5$ to $P_{95}$), every country-month, every metric and every threshold between the neighbors. 

In [ ]:
SCALES = {
    "linear": (lambda x: x, lambda y: y),
    "log": (lambda x: np.log(np.maximum(x, 1e-6)), np.exp),
    "log (loss + 0.0001)": (
        lambda x: np.log(x + T.LOSS_OFFSET),
        lambda y: np.exp(y) - T.LOSS_OFFSET,
    ),
}
THRESHOLDS = {
    m: sorted(
        set(fw_high.loc[fw_high.metric == m, "threshold"])
        | set(cfg.loc[cfg.metric == m, "threshold"])
        | set(fw_min.loc[fw_min.metric == m, "threshold"])
    )
    for m in T.METRICS
}
rows = []
for m in T.METRICS:
    for name, (f, finv) in SCALES.items():
        if (m == "loss") != (name == "log (loss + 0.0001)") and name != "linear":
            continue
        hits = cases = 0
        for lo, mid, hi in zip(T.PCTS[:-2], T.PCTS[1:-1], T.PCTS[2:]):
            a, b, actual = (df[f"{m}_p{p}"].to_numpy() for p in (lo, hi, mid))
            pred = finv(f(a) + (mid - lo) / (hi - lo) * (f(b) - f(a)))
            for t in THRESHOLDS[m]:
                between = (np.minimum(a, b) < t) & (np.maximum(a, b) > t)
                hits += ((pred[between] < t) == (actual[between] < t)).sum()
                cases += between.sum()
        rows.append((m, "log" if name != "linear" else "linear", hits / cases, cases))
held = pd.DataFrame(rows, columns=["metric", "scale", "correct side", "cases"]).pivot(
    index="metric", columns="scale", values="correct side"
)
held.loc[list(T.METRICS)].round(3)


The table reports the share of cases in which each scale places the hidden percentile on the correct side of the threshold.

| Metric | Linear | Log |
|---|---|---|
| Download | 0.792 | **0.872** |
| Upload | 0.761 | **0.852** |
| Latency | 0.741 | **0.764** |
| Packet loss | 0.643 | **0.837** |

Logarithmic interpolation is more accurate for every metric, although not in every individual case, as the above example shows. The gain is largest for packet loss. Pass rates are therefore interpolated on a log scale for all four metrics, with an offset of $10^{-4}$ for packet loss, which is often exactly zero. 

Following the held-out test above, we interpolate between adjacent percentiles on a log scale:

$$
F(x) = p_k + \frac{\ln x - \ln q_k}{\ln q_{k+1} - \ln q_k}\,(p_{k+1} - p_k), \qquad q_k \le x \le q_{k+1}
$$

where $q_k$ is the reported value at percentile level $p_k$ (e.g. $q = 52.33$ at $p = 0.25$). This assumes tests are spread evenly in *ratio* between adjacent percentiles: as many tests between 52 and 80 Mbps as between 80 and 122 Mbps.

For Andorra:

$$
F(100) = 0.25 + \frac{\ln 100 - \ln 52.33}{\ln 122.22 - \ln 52.33}\,(0.50 - 0.25) \approx 0.25 + 0.76 \times 0.25 \approx 0.44
$$

So an estimated 44% of tests fall below 100 Mbps, and the pass rate is $\pi(100) = 1 - F(100) \approx 0.56$.

### How to account for standard error? 

Andorra's 2001 tests are a sample. A different set of 2001 tests would give a slightly different pass rate. With $n$ tests, the standard error of the pass rate is

$$
\mathrm{SE} = \sqrt{\frac{\pi(1-\pi)}{n}}
$$

The score is reliable when the pass rate is far enough from 50% that sampling error is unlikely to change the outcome.

Writing $d = \lvert \pi - 0.5 \rvert$ for that distance, we require $d \ge z \cdot \mathrm{SE}$, where $z = 1.96$ for 95% confidence (two-sided).

Solving for $n$:

$$
n_{\text{req}} = \frac{z^2\,\pi(1-\pi)}{d^2}
$$

| Threshold | Pass rate $\pi$ | Distance $d$ | Tests needed | Andorra has |
|---|---|---|---|---|
| 100 Mbps | 0.56 | 0.06 | ~271 | 2001 |

Andorra's `download_p50` (122.22 Mbps) clears 100 Mbps by some margin, so about 271 tests are needed to be 95% confident it is not a chance result and that the score would come out the same if a different set of tests were run.


In [ ]:
# The Andorra example above, computed from the data
ad = df[(df.country_code == "AD") & (df.month == "2025-11-01")]
pi = T.pass_rate(ad, "download", 100)[0]
d = abs(pi - 0.5)
print(f"Andorra, November 2025: {ad.n_down.iloc[0]:,} tests")
print(f"F(100) = {1 - pi:.3f}, pass rate for download > 100 Mbps = {pi:.3f}, distance from 50% = {d:.3f}")
print(f"Tests needed: {T.Z**2 * pi * (1 - pi) / d**2:.0f}")

# How the number grows as the pass rate approaches 50% (worst case pi(1 - pi) = 0.25)
distance = np.array([0.20, 0.10, 0.05, 0.02, 0.01, 0.005])
pd.DataFrame(
    {"tests needed (95%)": np.ceil(T.Z**2 * 0.25 / distance**2).astype(int)},
    index=pd.Index([f"{x:.1%}" for x in distance], name="distance of the pass rate from 50%"),
)

The printout repeats the Andorra example from the data. *F(100)* is the estimated share of Andorra's November 2025 tests below 100 Mbit/s (44.1%), so the pass rate is 55.9%, 5.9 points from 50%, and 271 independent tests are enough to tell with 95% confidence that it is above 50%.

The table shows how that number grows as a pass rate gets closer to 50%. Each row is a distance from 50%, and the value is the number of independent tests needed to tell, with 95% confidence, which side of 50% the pass rate is on. It uses the worst case, π(1 − π) = 0.25, so it somewhat overstates the need when a pass rate is far from 50%. Halving the distance multiplies the tests needed by about four: 25 tests at 20 points, 97 at 10, 385 at 5 and 38,415 at half a point.

## How clustered are the tests? 

Tests are not independent: the same users test repeatedly, and a handful of networks may dominate in a given country. $n$ tests therefore carry the information of fewer independent ones. With only monthly percentile tables, the number of independent tests can be estimated from how pass rates move over time. 

We treat each country's monthly pass rate as a slowly moving true level plus independent sampling noise:

$$
\pi_t = \mu_t + \varepsilon_t
$$

Real change in $\mu_t$ tends to persist from one month to the next. Sampling noise $\varepsilon_t$ does not: a month that is too high by chance is followed by one that is, on average, back to normal.

Under this model, the covariance between consecutive changes equals minus the noise variance:

$$
\operatorname{Cov}(\Delta\pi_t,\ \Delta\pi_{t-1}) = -\sigma_\varepsilon^2
$$

Dividing this observed noise variance by the variance independent sampling would produce gives the design effect:

$$
\text{deff} = \frac{\sigma_\varepsilon^2}{\pi(1-\pi)/n}
$$

A design effect of 1 means tests behave as if independent. A value of 2 means the effective sample size is half of `sample_count`.

In [ ]:
tests = T.distinct_tests(cfg)
pr = T.pass_rates(df, tests)
pr[["cc", "month", "n_down", "n_up"]] = df[["country_code", "month", "n_down", "n_up"]]

rows = []
for cc, g in pr.sort_values("month").groupby("cc"):
    g = g.set_index("month").asfreq("MS")  # keep calendar spacing; gaps become NaN
    for k in tests.key:
        x = g[k]
        n = g["n_up"] if k.startswith("upload") else g["n_down"]
        d1 = x.diff()
        lag = (d1 * d1.shift(1)).dropna()
        p = x.mean()
        iid = (x * (1 - x) / n).mean()
        if len(lag) < 12 or not (0.05 < p < 0.95) or not iid > 0:
            continue
        rows.append((cc, k, g.n_down.median(), p, -lag.mean(), iid, x.var()))
dfx = pd.DataFrame(
    rows,
    columns=[
        "country_code",
        "test",
        "n_month",
        "pass_rate",
        "noise_var",
        "iid_var",
        "total_var",
    ],
)
dfx["deff"] = dfx.noise_var / dfx.iid_var
dfx["tests_per_month"] = pd.cut(
    dfx.n_month,
    [0, 1e3, 1e4, 1e5, np.inf],
    labels=["<1k", "1k–10k", "10k–100k", ">100k"],
)
summary = dfx.groupby("tests_per_month", observed=True).agg(
    series=("test", "size"),
    countries=("country_code", "nunique"),
    design_effect_q25=("deff", lambda s: s.quantile(0.25)),
    design_effect_median=("deff", "median"),
    design_effect_q75=("deff", lambda s: s.quantile(0.75)),
    estimate_negative=("deff", lambda s: (s < 0).mean()),
)
summary["independent-sampling share of monthly variance"] = dfx.groupby(
    "tests_per_month", observed=True
).apply(lambda x: (x.iid_var / x.total_var).median(), include_groups=False)
print(f"{len(dfx):,} monthly series, {dfx.country_code.nunique()} countries")
summary.round(2)


**series:** A series is one country's monthly pass rate for one requirement, tracked over the 24 months from January 2024 to December 2025. For example, Zambia's share of tests with download above 10 Mbit/s, month by month, is one series. There are 10 requirements, so a country can contribute up to 10 series.

For each series the notebook computes a design effect from how the pass rate jumps between months:

``noise_var`` is the month-to-month wobble that doesn't persist, i.e. random noise.
``iid_var`` is the wobble you'd expect if the tests were independent: π(1−π)/n.
The design effect is ``noise_var`` ÷ ``iid_var``. 

For example,Andorra's download above 10 Mbit/s is 

$$
0.000517 ÷ 0.000067 ≈ 7.7
$$ 

Its tests carry about as much information as one-eighth as many independent tests.

**tests_per_month:** The country's median number of download tests per month: under 1,000; 1,000–10,000; 10,000–100,000; over 100,000. A country falls in exactly one row.

**countries:** Number of unique countries

**design_effect_q25:** The 25th percentile of the design effects in the group.

**design_effect_median:** The 50th percentile of the design effects in the group.

**design_effect_q75:** The 75th percentile of the design effects in the group.

**estimate_negative:** The share of series whose estimate came out below zero. A real design effect can't be negative, since a variance can't be negative. A negative estimate means the random noise was too small to measure against real changes, such as a steady upward trend. Andorra's latency below 10 ms is an example (−5.1).

**independent-sampling share of monthly variance:** For the typical series, how much of the pass rate's month-to-month variation random sampling alone would explain. 0.11 means about 11%; 0.00 means essentially none.

The table shows that the design effect can only be measured where sampling noise is visible. In countries with fewer than about 10,000 tests a month, independent sampling accounts for a measurable share of the month-to-month movement, and the design effect there is about 2 to 4. In larger countries sampling noise is negligible: the movement is almost entirely real change or shifts in who is tested, so the estimate becomes erratic and is often negative. We therefore estimate the design effect from countries with fewer than 10,000 tests a month.

In [ ]:
identifiable = dfx[dfx.n_month < 10_000]
deff_hat = identifiable.deff.median()
print(
    f"Design effect where sampling noise is visible (< 10,000 tests a month): median {deff_hat:.1f} "
    f"(IQR {identifiable.deff.quantile(0.25):.1f}–{identifiable.deff.quantile(0.75):.1f}, {len(identifiable)} series)"
)
print(f"Value used in iqb_tools.DEFF: {T.DEFF}")
print("\nBy threshold (series with < 10,000 tests a month):")
print(identifiable.groupby("test").deff.median().round(1).to_string())

The first line gives the design effect we use from here on: the median over the 1,128 series from countries with fewer than 10,000 tests a month is 2.7, and the middle half of those series lie between 0.8 and 7.1 (the interquartile range, IQR). The second line shows the value stored in `iqb_tools.py`, 2.7 rounded up to 3.

The list gives the median design effect for each of the ten requirements on its own. Eight of the ten lie between 1.8 (latency below 10 ms) and 2.8, so rounding up to 3 covers them. The exceptions are latency below 50 ms (4.2) and upload above 25 Mbit/s (3.9), so no single requirement drives the overall estimate.

## How many tests are needed once tests are clustered?

The median design effect, 2.7, is rounded up to 3 (`T.DEFF`): three tests carry about as much information as one independent test.

We want every pass rate to be known to within ±5 percentage points with 95% confidence, whatever its value. The worst case is a pass rate of 50%, where $\pi(1-\pi)$ is largest, so

$$
n_{\min} = z^2 \cdot \frac{0.5 \times 0.5}{0.05^2} \cdot \text{deff} \approx 384 \times 3 \approx 1{,}153
$$

(rounded up). The count that matters is the smaller of the download and upload test counts (`T.binding_tests`): download, latency and packet-loss pass rates rest on download tests, and upload pass rates on upload tests, which are fewer. Income groups come from `outputs/tables/country_context.csv`, written by 0-setup.

In [ ]:
n_indep = T.Z**2 * 0.25 / T.TOL**2
print(f"Independent tests needed: {n_indep:.0f}; with design effect {T.DEFF}: {T.MIN_TESTS:,}")
display(
    pd.DataFrame(
        {f"design effect {k}": [int(np.ceil(n_indep * k))] for k in (1, 2, 3, 4, 10)},
        index=["minimum tests per period"],
    )
)

# Who meets the minimum in 2025, month by month and with the year pooled
ctx = T.read_table("country_context").set_index("country_code")
y25 = df[df.month.dt.year == 2025].assign(n=T.binding_tests)
by_country = y25.groupby("country_code").agg(
    months_meeting=("n", lambda n: (n >= T.MIN_TESTS).sum()),
    n_down=("n_down", "sum"),
    n_up=("n_up", "sum"),
)
by_country["year_meets"] = T.binding_tests(by_country) >= T.MIN_TESTS
by_country["income_group"] = (
    ctx.income_group.reindex(by_country.index).fillna("no WB income group")
)
GROUPS = T.INCOME_ORDER + ["no WB income group"]
meets = (
    by_country.groupby("income_group")
    .agg(
        countries=("year_meets", "size"),
        every_month=("months_meeting", lambda m: (m == 12).sum()),
        year_pooled=("year_meets", "sum"),
    )
    .reindex(GROUPS)
)
meets.loc["All"] = meets.sum()
meets.rename(
    index=T.INCOME_SHORT,
    columns={"every_month": "meet it every month", "year_pooled": "meet it with 2025 pooled"},
)

The printed line and the first table turn the design effect into a minimum number of tests. Without clustering (design effect 1), about 384 independent tests are needed for a pass rate to be known within ±5 points with 95% confidence (385 after rounding up). Each column multiplies that by a different design effect, so the minimum grows in proportion: 769 tests for a design effect of 2, 1,153 for 3 (the value we use), 1,537 for 4 and 3,842 for 10.

The second table counts, for each income group, the countries that meet the minimum of 1,153 tests in 2025:

| Column | Meaning |
|---|---|
| *countries* | The number of countries in the group with tests in 2025. |
| *meet it every month* | Countries with at least 1,153 tests in both directions in each of the 12 months. |
| *meet it with 2025 pooled* | Countries that reach 1,153 tests in both directions when all their 2025 tests are added together. |

The *no WB income group* row collects the 29 places without a World Bank income classification, mostly overseas territories and dependencies, plus Taiwan, the Holy See and Western Sahara. The share meeting the minimum every month rises with income, from 6 of 25 low-income countries (24%) to 60 of 85 high-income ones (71%). Pooling the year closes most of the gap: 22 of the 25 low-income countries then meet it. Overall, 138 of 245 meet it every month and 211 with the year pooled.

# Pooling where a reporting period falls short

A country below the minimum in a month can still get a score by pooling months. Pass rates combine exactly across months as a test-weighted average, so a pooled score is simply the score of all tests in the window.


We pool according to four guidelines:

1. **Pool over time.** 
2. **Use the shortest trailing window.** For each month, we use the shortest window ending in that month that has at least the minimum number of tests in both directions. A country with enough tests every month is never pooled; one that falls short only occasionally is pooled only in those months.
3. **Cap the window at 12 months.** Older tests describe a different network. If 12 months are not enough, there is no score.
4. **Label the window** next to the score, e.g. *IQB 0.6 (Oct–Dec 2025)*.

In [ ]:
MAX_WINDOW = 12
months = pd.date_range("2024-01-01", "2025-12-01", freq="MS")
y25_months = [mo for mo in months if mo.year == 2025]
nd = df.pivot(index="country_code", columns="month", values="n_down").reindex(columns=months).fillna(0)
nu = df.pivot(index="country_code", columns="month", values="n_up").reindex(columns=months).fillna(0)


def trailing_window(month):
    """Shortest trailing window (in months) ending at `month` with >= MIN_TESTS in both directions."""
    j = months.get_loc(month)
    cd = nd.iloc[:, : j + 1].iloc[:, ::-1].cumsum(axis=1).to_numpy()  # tests in the last 1, 2, 3, ... months
    cu = nu.iloc[:, : j + 1].iloc[:, ::-1].cumsum(axis=1).to_numpy()
    ok = (cd >= T.MIN_TESTS) & (cu >= T.MIN_TESTS)
    k = np.where(ok.any(axis=1), ok.argmax(axis=1) + 1, np.inf)
    return np.where(k <= MAX_WINDOW, k, np.inf)


Wn = pd.DataFrame({mo: trailing_window(mo) for mo in y25_months}, index=nd.index)  # window per country-month
plan = pd.DataFrame(
    {
        "median window (months)": Wn.median(axis=1),
        "longest window (months)": Wn.max(axis=1),
        "months pooled": (Wn > 1).sum(axis=1),
        "months without a score": np.isinf(Wn).sum(axis=1),
    }
).join(ctx[["country", "income_group"]])

PLANS = [
    "monthly, never pooled",
    "monthly, pooled in some months",
    "pooled over 2–3 months",
    "pooled over 4–6 months",
    "pooled over 7–12 months",
    "no score in most months",
]


def label(r):
    if r["months without a score"] >= 6:
        return PLANS[5]
    if r["longest window (months)"] == 1:
        return PLANS[0]
    if r["median window (months)"] <= 1:
        return PLANS[1]
    if r["median window (months)"] <= 3:
        return PLANS[2]
    return PLANS[3] if r["median window (months)"] <= 6 else PLANS[4]


plan["plan"] = plan.apply(label, axis=1)

by_inc = (
    pd.crosstab(plan.plan, plan.income_group.fillna("no WB income group"))
    .reindex(index=PLANS, columns=GROUPS)
    .fillna(0)
    .astype(int)
)
by_inc["All"] = by_inc.sum(axis=1)
by_inc.rename(columns=T.INCOME_SHORT)

The code finds, for every month of 2025, the shortest window ending in that month with the minimum number of tests, and then places each country in one category (rows) according to how its windows behave over the year. The table counts the countries in each category by income group (columns):

| Category | Meaning |
|---|---|
| *monthly, never pooled* | Every month of 2025 has the minimum on its own. |
| *monthly, pooled in some months* | Most months have the minimum on their own, but at least one month has to be pooled. |
| *pooled over 2–3 months*, *4–6 months*, *7–12 months* | Pooling is routine: the median window over the 12 months is 2–3, 4–6 or 7–12 months long. |
| *no score in most months* | In at least 6 of the 12 months, even 12 months of tests are not enough. |

The number of countries that never need pooling rises with income: 6 of 25 low-income, 26 of 47 lower-middle-income, 41 of 59 upper-middle-income and 60 of 85 high-income countries. Most places without a World Bank income group are in the last row (18 of 29).

In [ ]:
for p in PLANS[1:]:
    x = plan[plan.plan == p].sort_values(["median window (months)", "country"])
    print(f"{p} ({len(x)}):")
    print("   " + ", ".join(x.country) + "\n")

The lists name the countries in every category except *monthly, never pooled*, sorted by their median window and then by name, which is why they are not strictly alphabetical. Their lengths match the *All* column of the table (14, 35, 18, 8 and 32).

The table shows who needs pooling in 2025. In 138 countries every month has the minimum in both directions, so they never need it. Another 14 need it only occasionally, when one or two months fall short (for example Mongolia, Iran, Zambia, Malawi and Rwanda).

Pooling is routine for 61 countries: 35 typically reach the minimum with 2–3 months, 18 with 4–6 and 8 with 7–12. 
They are mostly small states and territories (Andorra, Monaco, Barbados, the Virgin Islands) and low- and lower-middle-income countries such as Afghanistan, Niger, Mali, DR Congo, Haiti, Tajikistan, South Sudan, Chad and Burundi. The remaining 32 places cannot reach the minimum in most months even with 12 months of tests. Almost all are small islands and territories, plus the Central African Republic, Comoros, Guinea-Bissau and North Korea. 

Low-income countries are the most affected group: only 6 of 25 reach the minimum every month, 12 need routine pooling and 3 get no score in most months. With the whole year pooled, 22 of the 25 meet the guidelines.

Pooling has two costs. A pooled score describes the whole window, so a sudden real change (an outage, a new cable) is blurred and reported late; for countries pooled most months, a fixed quarterly or half-yearly reporting period is easier to read than a window whose length changes from month to month. And because the same users test across months, pooling may add fewer independent tests than the count suggests.

## Save

In [ ]:
dfx.to_csv(T.TAB / "design_effect_series.csv", index=False)
plan.to_csv(T.TAB / "pooling_plan.csv")
print("wrote design_effect_series.csv", dfx.shape, "and pooling_plan.csv", plan.shape, "to", T.TAB)

## Summary

**We set out to find how many tests a country needs before its IQB score can be displayed with confidence.** Because IQB only asks whether each requirement passes, the score at P50 depends on whether each pass rate is above or below 50%. The closer a pass rate sits to 50%, the more tests are needed to tell which side it falls on: 25 independent tests when it is 20 points away, 385 at 5 points, and 38,415 at half a point.

**We recover pass rates from the nine published percentiles by interpolating on a log scale.** In the held-out test, log interpolation placed the hidden percentile on the correct side of a threshold more often than linear interpolation for all four metrics, most clearly for packet loss (0.837 against 0.643).

**Tests are clustered, because the same users and networks test repeatedly.** Estimated from how monthly pass rates move, the design effect is a median of 2.7 in countries where sampling noise is visible, which we round to 3. Requiring every pass rate to be known to within ±5 percentage points with 95% confidence then gives a minimum of 1,153 tests per reporting period, counted on the smaller of download and upload.

**In 2025, 138 of the 245 countries meet that minimum every month, and 211 meet it with the whole year pooled.** Monthly reporting needs pooling for 75 countries, 14 occasionally and 61 routinely, and 32 places, almost all small islands and territories, cannot reach the minimum in most months even with 12 months of tests. Low-income countries are the most affected: only 6 of 25 meet the minimum every month, although 22 do with the year pooled.